# Решение · мини-проект Hymenoptera (пчёлы / муравьи)

Открывай **после** попытки в [handbook_gl42_практика.ipynb](./handbook_gl42_практика.ipynb).

Источник handbook: [transfer-learning.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/transfer-learning.html) · теория [§6](../d.5.42%20Transfer%20learning.md#6-практика-handbook--мини-проект).

**🟠 Дубинин:** тот же датасет и идея transfer — [видео 23](https://www.youtube.com/watch?v=M0iB56Vwy64&t=344) (плейлист [PyTorch](https://www.youtube.com/playlist?list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1)), но в ролике **своя CNN** (`features`/`classifier`), без `ResNet.layer4`. Этот ноутбук = **версия handbook / туториала** на **ResNet-18**.

Код по [официальному туториалу PyTorch](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html), упрощён: два прогона **шаг 1** (только `fc`) и **шаг 2** (+ `layer4`, `lr=1e-4`). Пояснение «почему fine-tune не всегда лучше» — **последняя markdown-ячейка** внизу.

**Данные:** при первом запуске ячейки ниже — **автоскачивание** [hymenoptera_data.zip](https://download.pytorch.org/tutorial/hymenoptera_data.zip) в `practice/data/` (как в туториале PyTorch).


## Импорты и путь к данным

In [7]:
import copy
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("data")
DATA_DIR = DATA_ROOT / "hymenoptera_data"
HYMENOPTERA_ZIP_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
NUM_CLASSES = 2
BATCH_SIZE = 8
NUM_EPOCHS_FE = 5   # шаг 1: feature extraction (увеличь до ~15–25 как в туториале)
NUM_EPOCHS_FT = 5   # шаг 2: fine-tune layer4


def ensure_hymenoptera_data():
    if (DATA_DIR / "train").is_dir() and (DATA_DIR / "val").is_dir():
        return
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / "hymenoptera_data.zip"
    if not zip_path.is_file():
        print("Скачиваю hymenoptera_data.zip (~45 МБ)...")
        urllib.request.urlretrieve(HYMENOPTERA_ZIP_URL, zip_path)
    print("Распаковка...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_ROOT)
    if not (DATA_DIR / "train").is_dir():
        raise RuntimeError(f"После распаковки нет {DATA_DIR / 'train'}. Проверь архив.")


ensure_hymenoptera_data()
print("device:", device)
print("data:", DATA_DIR.resolve())


device: cpu
data: C:\Users\Иван\Desktop\conspects_ml\d.Машинное обучение\d.5 advanced Deep Learning\d.5.42 Transfer learning\practice\data\hymenoptera_data


## Шаг 3 handbook · transforms (аугментации + ImageNet Normalize)

Используем с **начала** обучения (в туториале так же).

In [8]:
data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

image_datasets = {
    x: datasets.ImageFolder(DATA_DIR / x, data_transforms[x])
    for x in ["train", "val"]
}
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == "train"), num_workers=0)
    for x in ["train", "val"]
}
class_names = image_datasets["train"].classes
print("classes:", class_names, "| train:", len(image_datasets["train"]), "val:", len(image_datasets["val"]))


classes: ['ants', 'bees'] | train: 244 val: 153


## Цикл обучения (как в туториале PyTorch)

**После каждой эпохи** в логе две строки:

| Строка | Что значит |
|--------|------------|
| `train loss` / `train acc` | ошибка и **доля верных** ответов на **обучающих** картинках (веса меняются) |
| `val loss` / `val acc` | то же на **валидации** (веса **не** меняются, только проверка) |
| `best val acc` в конце этапа | **лучший** val acc за все эпохи — по нему сохраняются веса модели |

`acc=0.9412` → **94.12%** картинок классифицированы правильно (пчела/муравей).

In [9]:
def train_model(model, criterion, optimizer, num_epochs, phase_name=""):
    since = time.time()
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print(f"\n{phase_name} epoch {epoch + 1}/{num_epochs}")
        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])
            tag = "обучение" if phase == "train" else "проверка val"
            print(f"  [{tag}] loss={epoch_loss:.4f}  acc={epoch_acc:.4f}  ({epoch_acc * 100:.1f}% верно)")
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

    elapsed = time.time() - since
    print(f"{phase_name} — ИТОГ: лучшая val accuracy = {best_acc:.4f} ({best_acc * 100:.1f}% верно), время {elapsed:.0f}s")
    model.load_state_dict(best_wts)
    return model, float(best_acc)


## Шаг 1 handbook · Feature extraction (только `fc`)

**После ячейки смотри:**
- `trainable (шаг 1): ['fc.weight', 'fc.bias']` — учится **только голова**, ResNet заморожен.
- **ИТОГ** с `best val accuracy` — метрика handbook **для шага 1** (запиши число).

In [10]:
def make_resnet_feature_extract():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(device)

model_fe = make_resnet_feature_extract()
criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(filter(lambda p: p.requires_grad, model_fe.parameters()), lr=1e-3)

trainable = [n for n, p in model_fe.named_parameters() if p.requires_grad]
print("trainable (шаг 1):", trainable)

model_fe, acc_fe = train_model(model_fe, criterion, optimizer_fe, NUM_EPOCHS_FE, "[шаг 1 FE]")
print(f"\n>>> Запомни для отчёта: шаг 1 · только fc · val accuracy = {acc_fe:.4f} ({acc_fe * 100:.1f}%)")


trainable (шаг 1): ['fc.weight', 'fc.bias']

[шаг 1 FE] epoch 1/5
  [обучение] loss=0.5689  acc=0.6885  (68.9% верно)
  [проверка val] loss=0.4051  acc=0.8758  (87.6% верно)

[шаг 1 FE] epoch 2/5
  [обучение] loss=0.4394  acc=0.8074  (80.7% верно)
  [проверка val] loss=0.2962  acc=0.9150  (91.5% верно)

[шаг 1 FE] epoch 3/5
  [обучение] loss=0.3235  acc=0.8934  (89.3% верно)
  [проверка val] loss=0.2382  acc=0.9216  (92.2% верно)

[шаг 1 FE] epoch 4/5
  [обучение] loss=0.2790  acc=0.9098  (91.0% верно)
  [проверка val] loss=0.2155  acc=0.9412  (94.1% верно)

[шаг 1 FE] epoch 5/5
  [обучение] loss=0.3096  acc=0.8770  (87.7% верно)
  [проверка val] loss=0.2284  acc=0.9085  (90.8% верно)
[шаг 1 FE] — ИТОГ: лучшая val accuracy = 0.9412 (94.1% верно), время 53s

>>> Запомни для отчёта: шаг 1 · только fc · val accuracy = 0.9412 (94.1%)


## Шаг 2 handbook · Fine-tune `layer4` (`lr=1e-4`)

Продолжаем с весами после шага 1 (голова уже обучена).

**После ячейки смотри:**
- `trainable (шаг 2): 17 тензоров` — учится **голова + последний блок** ResNet (`layer4`).
- сравнение **двух ИТОГОВ** val accuracy — что просит handbook.

In [11]:
model_ft = copy.deepcopy(model_fe)
for p in model_ft.layer4.parameters():
    p.requires_grad = True

optimizer_ft = optim.Adam([
    {"params": model_ft.fc.parameters(), "lr": 1e-3},
    {"params": model_ft.layer4.parameters(), "lr": 1e-4},
])

trainable = [n for n, p in model_ft.named_parameters() if p.requires_grad]
print("trainable (шаг 2):", len(trainable), "тензоров")

model_ft, acc_ft = train_model(model_ft, criterion, optimizer_ft, NUM_EPOCHS_FT, "[шаг 2 FT]")

delta = acc_ft - acc_fe
print("\n" + "=" * 56)
print("СРАВНЕНИЕ (handbook)")
print(f"  шаг 1 · только fc        : val accuracy = {acc_fe:.4f}  ({acc_fe * 100:.1f}%)")
print(f"  шаг 2 · fc + layer4      : val accuracy = {acc_ft:.4f}  ({acc_ft * 100:.1f}%)")
print(f"  разница (шаг2 − шаг1)    : {delta:+.4f}")
if delta > 0:
    print("  вывод: fine-tune layer4 улучшил val — записывай оба числа.")
elif delta < 0:
    print("  вывод: fine-tune НЕ обогнал «только fc» — это нормально (см. текст в конце ноутбука).")
else:
    print("  вывод: одинаково.")
print("=" * 56)


trainable (шаг 2): 17 тензоров

[шаг 2 FT] epoch 1/5
  [обучение] loss=0.4102  acc=0.8402  (84.0% верно)
  [проверка val] loss=0.2017  acc=0.9412  (94.1% верно)

[шаг 2 FT] epoch 2/5
  [обучение] loss=0.2618  acc=0.8811  (88.1% верно)
  [проверка val] loss=0.1847  acc=0.9281  (92.8% верно)

[шаг 2 FT] epoch 3/5
  [обучение] loss=0.1441  acc=0.9467  (94.7% верно)
  [проверка val] loss=0.1635  acc=0.9542  (95.4% верно)

[шаг 2 FT] epoch 4/5
  [обучение] loss=0.2052  acc=0.9098  (91.0% верно)
  [проверка val] loss=0.1803  acc=0.9412  (94.1% верно)

[шаг 2 FT] epoch 5/5
  [обучение] loss=0.2322  acc=0.9057  (90.6% верно)
  [проверка val] loss=0.2771  acc=0.9216  (92.2% верно)
[шаг 2 FT] — ИТОГ: лучшая val accuracy = 0.9542 (95.4% верно), время 77s

СРАВНЕНИЕ (handbook)
  шаг 1 · только fc        : val accuracy = 0.9412  (94.1%)
  шаг 2 · fc + layer4      : val accuracy = 0.9542  (95.4%)
  разница (шаг2 − шаг1)    : +0.0131
  вывод: fine-tune layer4 улучшил val — записывай оба числа.


## Справка · что видно после обучения

```text
trainable (шаг 1)     → какие веса вообще могут обновляться
[обучение] / [проверка val]  → train = учим; val = только смотрим качество
ИТОГ: лучшая val accuracy    → главная цифра этапа (0.94 ≈ 94% угадано)
СРАВНЕНИЕ (handbook)        → финальная строка отчёта: шаг1 vs шаг2
```

## Что проверить · интерпретация

Handbook просит **сравнить** шаг 1 и шаг 2 — не обязательно, что fine-tune всегда **выше**.

| Наблюдение | Смысл |
|------------|--------|
| Шаг 1 уже **~0.94** val acc | ImageNet-признаки + маленькая голова часто хватает на пчёл/муравьёв |
| Шаг 2 **чуть ниже** (напр. 0.9412 → 0.9346) | Нормально: двигаем **претрейн** `layer4` на **~150 val** картинках, риск переобучения / шума |
| `trainable (шаг 1)`: только `fc` | Заморозка сработала |
| `trainable (шаг 2)`: 17 тензоров | `fc` + веса `layer4` |

**Если нужно приблизиться к туториалу:** `NUM_EPOCHS_FE/FT` ~ **15–25**, GPU, для шага 2 уменьши `lr` у головы (напр. `fc` → `1e-4`).

**Для сдачи handbook:** зафиксируй обе цифры val acc и одной фразой — «fine-tune layer4 дал прирост / не дал прирост и почему (мало данных, голова уже насытилась)».


---

# Почему fine-tune не всегда лучше

**Где эта ячейка:** самый низ ноутбука — последняя markdown после ячейки «шаг 2» и блока «Справка».

Transfer learning — это **не** «обязательно покрутить ещё слои сети». Обычно два уровня:

1. **Feature extraction (шаг 1)** — тело ResNet с ImageNet **заморожено**, учим только **голову** `fc`. Уже перенос: признаки не с нуля.
2. **Fine-tuning (шаг 2)** — *по желанию* чуть двигаем `layer4`, если хотим выжать ещё val accuracy.

**Fine-tune не обязан улучшить результат.** Часто шаг 1 уже даёт высокий val. Тогда шаг 2 может дать **примерно то же** или **чуть хуже** — как у тебя: ~**94.1%** (только `fc`) и ~**93.5%** (`fc` + `layer4`). Мы **добавили** fine-tune, метрика **чуть упала** — при рабочем коде это **нормально**.

**Почему:** ImageNet-признаки хорошо подходят к пчёлам/муравьям; голова быстро настраивается. Когда крутим `layer4` на **~244 train / ~153 val** картинках и **5 эпохах**, легко **слегка испортить** удачные претрейн-веса (шум, переобучение на train).

**Fine-tune чаще помогает**, если только голова дала **низкий** val; задача **далека** от ImageNet; **больше** данных и эпох; **очень маленький** `lr` на тело.

**Часто достаточно шага 1**, если val после головы уже **>90%** — как в этом датасете.

**Что записать в отчёт handbook:** две цифры val + одна фраза: «остановился бы на feature extraction» или «fine-tune оправдан, потому что …».
